In [2]:
import cv2
import numpy as np
import os
import pandas as pd
import scanpy as sc
import tifffile
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle

In [3]:
# Adaptive thresholding with size filter and dilation
def adaptive_thresholding_with_size_filter_and_dilation(img, block_size=49, c=-1, min_area=100, radius=10):
    th = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block_size, c)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filtered_contours = [c for c in contours if cv2.contourArea(c) >= min_area]
    filtered_img = np.zeros_like(th)
    cv2.drawContours(filtered_img, filtered_contours, -1, 255, -1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*radius+1, 2*radius+1))
    th_dilated = cv2.dilate(filtered_img, kernel)
    return th_dilated

In [4]:
# Settings
settings = {
    # "MERSCOPE_2m_WT_R1": {"pixel_size": 0.10799861, "x_shift": int(-266.1734), "y_shift": int(180.2510), "cutoff": 6250, "theta": 10 * np.pi / 180, "coordinate_for_rotation": ["global_y", "global_x"], "coordinate_for_cutoff": "global_y", "cutoff_direction": "smaller"},
    # "MERSCOPE_2m_AD_R1": {"pixel_size": 0.10799905, "x_shift": int(-126.9911), "y_shift": int(-20.3805), "cutoff": -4200, "theta": 170 * np.pi / 180, "coordinate_for_rotation": ["global_x", "global_y"], "coordinate_for_cutoff": "global_x", "cutoff_direction": "larger"},
    "MERSCOPE_9m_AD_R1": {"pixel_size": 0.1079991995, "x_shift": int(-17.2801), "y_shift": int(277.6227), "cutoff": -6000, "theta": 200 * np.pi / 180, "coordinate_for_rotation": ["global_x", "global_y"], "coordinate_for_cutoff": "global_x", "cutoff_direction": "smaller"},
    "MERSCOPE_9m_AD_R2": {"pixel_size": 0.10799904, "x_shift": int(-31.3303), "y_shift": int(268.1251), "cutoff": -5850, "theta": 190 * np.pi / 180, "coordinate_for_rotation": ["global_x", "global_y"], "coordinate_for_cutoff": "global_x", "cutoff_direction": "larger"},
    "MERSCOPE_9m_WT_R1": {"pixel_size": 0.10799939, "x_shift": int(-53.0503), "y_shift": int(29.6203), "cutoff": -5450, "theta": 185 * np.pi / 180, "coordinate_for_rotation": ["global_x", "global_y"], "coordinate_for_cutoff": "global_x", "cutoff_direction": "smaller"},
    "MERSCOPE_9m_WT_R2": {"pixel_size": 0.10799863, "x_shift": int(145.7568), "y_shift": int(-127.0323), "cutoff": -5850, "theta": 190 * np.pi / 180, "coordinate_for_rotation": ["global_x", "global_y"], "coordinate_for_cutoff": "global_x", "cutoff_direction": "larger"},
    }

In [6]:
radius = 5
in_soma_label = f"overlaps_nucleus_{radius}_dilation"

In [ ]:
# ==================== Main operations on transcripts ==================== #

for dataset in settings.keys():
    
    print("=" * 25)
    print(f"Processing {dataset}...")
    print("=" * 25)
    
    # File paths
    data_path = f"../data/{dataset}/"
    output_path = f"../output/{dataset}/"

    # Transformation parameters
    pixel_size = settings[dataset]["pixel_size"]
    x_shift = settings[dataset]["x_shift"]
    y_shift = settings[dataset]["y_shift"]
    
    # Load DAPI images
    files = os.listdir(data_path + "raw_data/DAPI_images/")
    files = [i for i in files if i.startswith("mosaic_DAPI_z")]
    files.sort()
    
    # # Compute all images stack
    # all_images = []
    # for fname in files:
    #     img = tifffile.imread(data_path + f"raw_data/DAPI_images/{fname}")
    #     all_images.append(img)
    # all_images_stack = np.stack(all_images)
    
    # Read transcripts
    transcripts = pd.read_csv(data_path + "raw_data/transcripts.csv")
    transcripts = transcripts[["cell_id", "gene", "global_x", "global_y", "global_z"]].copy()
    
    # Compute DAPI pixel coordinates
    transcripts["row"] = (transcripts["global_y"] / pixel_size).astype(int) + y_shift
    transcripts["col"] = (transcripts["global_x"] / pixel_size).astype(int) + x_shift

    # Add default overlap column
    transcripts[in_soma_label] = 0
    
    # Update labels in place
    global_ratio = []

    print(f"Processing dilation radius: {radius}")

    for j, fname in enumerate(files):
        
        # Load DAPI image
        img = tifffile.imread(data_path + f"raw_data/DAPI_images/{fname}")
        
        if dataset == "MERSCOPE_9m_WT_R1":
            p99 = np.percentile(img, 99.9)
            img = np.clip(img, 0, p99)
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        
        # Threshold and dilate
        th = adaptive_thresholding_with_size_filter_and_dilation(img, block_size=49, c=-1, min_area=100, radius=radius)
        
        # Select transcripts in this z-layer
        trans_z_mask = transcripts["global_z"] == j
        trans_z = transcripts[trans_z_mask].copy()
        row_vals = trans_z["row"].astype(int).values
        col_vals = trans_z["col"].astype(int).values

        # Avoid out-of-bounds indexing
        height, width = th.shape
        valid = (row_vals >= 0) & (row_vals < height) & (col_vals >= 0) & (col_vals < width)
        row_valid = row_vals[valid]
        col_valid = col_vals[valid]
        
        # Assign in-nucleus labels
        overlaps = np.zeros(len(trans_z), dtype=int)
        overlaps[valid] = (th[row_valid, col_valid] != 0).astype(int)

        # Update main DataFrame in-place
        transcripts.loc[trans_z.index, in_soma_label] = overlaps

        # Track global ratio

    transcripts = transcripts.rename(columns = {"gene": "target"})
    transcripts.to_parquet(data_path + "intermediate_data/transcripts_tmp.parquet")
    print(f"Processed {dataset}.")

In [7]:
# ==================== Postprocessing on transcripts ==================== #

for dataset in settings.keys():
    
    print("=" * 25)
    print(f"Processing {dataset}...")
    print("=" * 25)
    
    # File paths
    data_path = f"../data/{dataset}/"
    output_path = f"../output/{dataset}/"
    
    # Read data
    transcripts = pd.read_parquet(data_path + "intermediate_data/transcripts_tmp.parquet")
    
    # Cut hemisphere
    cutoff = settings[dataset]["cutoff"]
    theta = settings[dataset]["theta"]
    coordinate_for_rotation = settings[dataset]["coordinate_for_rotation"]
    coordinate_for_cutoff = settings[dataset]["coordinate_for_cutoff"]
    cutoff_direction = settings[dataset]["cutoff_direction"]
    
    rotation_matrix = np.array([[np.cos(theta), np.sin(theta)], [-np.sin(theta), np.cos(theta)]])
    coords = transcripts[coordinate_for_rotation].to_numpy()
    transformed_coords = coords @ rotation_matrix.T
    transcripts[f"{coordinate_for_rotation[0]}_new"] = transformed_coords[:, 0]
    transcripts[f"{coordinate_for_rotation[1]}_new"] = transformed_coords[:, 1]
    if cutoff_direction == "smaller":
        transcripts_cut = transcripts[transcripts[f"{coordinate_for_cutoff}_new"] <= cutoff].copy()
    elif cutoff_direction == "larger":
        transcripts_cut = transcripts[transcripts[f"{coordinate_for_cutoff}_new"] >= cutoff].copy()
    transcripts_cut = transcripts_cut[["cell_id", in_soma_label, "target", "global_x", "global_y", "global_z"]].copy()
    transcripts_cut.rename(columns={in_soma_label: "overlaps_nucleus"}, inplace=True)

    transcripts_cut.to_parquet(data_path + "processed_data/transcripts.parquet")
    print(f"Processed {dataset}.")

Processing MERSCOPE_9m_AD_R1...
Processed MERSCOPE_9m_AD_R1.
Processing MERSCOPE_9m_AD_R2...
Processed MERSCOPE_9m_AD_R2.
Processing MERSCOPE_9m_WT_R1...
Processed MERSCOPE_9m_WT_R1.
Processing MERSCOPE_9m_WT_R2...
Processed MERSCOPE_9m_WT_R2.
